In [2]:
from jaad_data import JAAD

jaad_api = JAAD(data_path = '.')

In [2]:
# Extract & Save Images
# JAAD has in-built method extract_and_save_images, but it extracts images in png format, which takes up too much disk space.
import os
import cv2

def extract_all_images(jaad_obj, image_ext=".jpg", overwrite=False):
    clip_files = sorted(
        f for f in os.listdir(jaad_obj._clips_path)
        if f.lower().endswith(".mp4")
    )

    os.makedirs(jaad_obj._images_path, exist_ok=True)
    print(f"Found {len(clip_files)} clips in {jaad_obj._clips_path}")

    for i, clip_file in enumerate(clip_files, start=1):
        vid = os.path.splitext(clip_file)[0]
        clip_path = os.path.join(jaad_obj._clips_path, clip_file)
        save_dir = os.path.join(jaad_obj._images_path, vid)
        os.makedirs(save_dir, exist_ok=True)

        cap = cv2.VideoCapture(clip_path)
        if not cap.isOpened():
            print(f"[{i}/{len(clip_files)}] [SKIP] cannot open: {clip_path}")
            continue

        frame_idx = 0
        saved = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            out_path = os.path.join(save_dir, f"{frame_idx:05d}{image_ext}")
            if overwrite or not os.path.exists(out_path):
                cv2.imwrite(out_path, frame)
                saved += 1
            frame_idx += 1

        cap.release()
        print(f"[{i}/{len(clip_files)}] {vid}: saved {saved} / {frame_idx} frames")

# Run extraction for all videos
extract_all_images(jaad_api, image_ext=".jpg", overwrite=False)

Found 346 clips in .\JAAD_clips
[1/346] video_0001: saved 0 / 600 frames
[2/346] video_0002: saved 0 / 210 frames
[3/346] video_0003: saved 0 / 210 frames
[4/346] video_0004: saved 0 / 180 frames
[5/346] video_0005: saved 0 / 240 frames


KeyboardInterrupt: 

In [3]:
# Generate Database
db = jaad_api.generate_database()

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\ASUS\Documents\Year 3 Semester 2\JAAD\data_cache\jaad_database.pkl


In [4]:
# Extract Features from Pedestrians
# Notes on meaning of Behavior Annotations:
    # occlusion: 0(not occluded), 1(partially occluded), 2(fully occluded)
    # cross: 0(not crossing), 1(crossing)
    # reaction: 0(no reaction), 1(reaction)
    # hand_gesture: 0(no hand gesture), 1(hand gesture)
    # look: 0(not looking), 1(looking)
    # action: 0(Standing), 1(Walking)
    # nod: 0(no nod), 1(nod)
# Notes on meaning of Pedestrian Attributes:
    # old_id: Original Annotation ID String
    # age: 0(child), 1 (young), 2 (adult), 3 (senior)
    # crossing: 0(not crossing), 1(crossing), -1(irrelevant)
    # crossing_point: Frame Index of Crossing Point (if crossing), -1 otherwise
    # decision_point: Frame Index of Decision Point (if crossing), -1 otherwise
    # designated: 0(not designated crossing point), 1(designated crossing point)
    # gender: 0(n/a), 1(female), 2(male)
    # group_size: Number of People in Group
    # intersection: 0(not at intersection), 1(at intersection)
    # motion_direction: 0(n/a), 1(Lateral / Across), 2(Longitudinal / Along)
    # num_lanes: Number of Road Lanes
    # signalized: 0(n/a), 1(non-signalized intersection), 2(signalized intersection)
    # traffic_direction: 0(One-Way), 1(Two-Way)

pedestrian_ids = jaad_api._get_pedestrian_ids()

features = []
for vid, video in db.items():
    for pid, pedestrian in video["ped_annotations"].items():
        if 'b' not in pid: # Skip Pedestrians without Behavior Annotations
            continue
        
        # Dynamic Features (Time-Series)
        frames = pedestrian.get("frames", [])
        bboxes = pedestrian.get("bbox", [])
        occlusions = pedestrian.get("occlusion", [])
        behavior = pedestrian.get("behavior", {})

        # Static Features (Non-Time-Series)
        attributes = pedestrian.get("attributes", {})

        for i, frame in enumerate(frames):
            row = {
                "video_id": vid,
                "pedestrian_id": pid,

                # Frames
                "frame_id": frame,

                # BBoxes
                "bbox_x1": bboxes[i][0],
                "bbox_y1": bboxes[i][1],
                "bbox_x2": bboxes[i][2],
                "bbox_y2": bboxes[i][3],

                # Occlusions
                "occlusion": occlusions[i],

                # Behavior Annotations
                "cross": behavior.get("cross", [0]*len(frames))[i],
                "reaction": behavior.get("reaction", [0]*len(frames))[i],
                "hand_gesture": behavior.get("hand_gesture", [0]*len(frames))[i],
                "look": behavior.get("look", [0]*len(frames))[i],
                "action": behavior.get("action", [0]*len(frames))[i],
                "nod": behavior.get("nod", [0]*len(frames))[i],

                # Pedestrian Attributes
                "old_id": attributes.get("old_id", ""),
                "age": attributes.get("age", 0),
                "crossing": attributes.get("crossing", 0),
                "crossing_point": attributes.get("crossing_point", 0),
                "decision_point": attributes.get("decision_point", 0),
                "designated": attributes.get("designated", 0),
                "gender": attributes.get("gender", 0),
                "group_size": attributes.get("group_size", 1),
                "intersection": attributes.get("intersection", 0),
                "motion_direction": attributes.get("motion_direction", 0),
                "num_lanes": attributes.get("num_lanes", 0),
                "signalized": attributes.get("signalized", 0),
                "traffic_direction": attributes.get("traffic_direction", 0)
            }
            features.append(row)


import pandas as pd

df = pd.DataFrame(features)
print(f"Features shape: {df.shape}")
display(df.head())

# Count number of unique pedestrians
unique_pedestrians = df["pedestrian_id"].nunique()
assert unique_pedestrians == len([pid for pid in pedestrian_ids if 'b' in pid]), "Mismatch between unique pedestrians in features and pedestrian IDs."

# Check for Class Imbalance in Crossing Behavior
crossing_counts = df["crossing"].value_counts()
display(crossing_counts)

---------------------------------------------------------
Generating database for jaad
jaad database loaded from c:\Users\ASUS\Documents\Year 3 Semester 2\JAAD\data_cache\jaad_database.pkl
Features shape: (132700, 27)


,video_id,pedestrian_id,frame_id,bbox_x1,bbox_y1,bbox_x2,bbox_y2,occlusion,cross,reaction,...,crossing_point,decision_point,designated,gender,group_size,intersection,motion_direction,num_lanes,signalized,traffic_direction
0,video_0001,0_1_3b,0,465.0,730.0,533.0,848.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
1,video_0001,0_1_3b,1,463.0,730.0,532.0,848.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
2,video_0001,0_1_3b,2,461.0,730.0,531.0,849.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
3,video_0001,0_1_3b,3,459.0,730.0,530.0,849.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1
4,video_0001,0_1_3b,4,458.0,731.0,530.0,851.0,0,0,0,...,-1,363,0,1,1,0,2,2,0,1


crossing
 1    105026
-1     16216
 0     11458
Name: count, dtype: int64

In [39]:
# Train XGBoost classifier to predict crossing behavior (frame-level binary classification)

import os
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_curve, classification_report, roc_auc_score
from xgboost import XGBClassifier

# Safety-focused threshold shared by evaluation and video rendering
DECISION_THRESHOLD = 0.25

# Keep names consistent for training/inference/video rendering
cat_cols = [
    "occlusion", "reaction", "hand_gesture", "look", "action", "nod",
    "age", "designated", "gender", "intersection",
    "motion_direction", "signalized", "traffic_direction"
]
num_cols = [
    "bbox_center_x", "bbox_center_y", "bbox_width", "bbox_height", "bbox_area",
    "velocity_x", "velocity_y", "speed", "acceleration_x", "acceleration_y",
    "group_size", "num_lanes"
]


def engineer_features(frame_df):
    """Compute derived geometry and motion features used by the model."""
    out = frame_df.copy()

    # JAAD frames are 1920 x 1080
    W, H = 1920.0, 1080.0

    # Normalized bbox geometry
    out["bbox_center_x"] = ((out["bbox_x1"] + out["bbox_x2"]) / 2.0) / W
    out["bbox_center_y"] = ((out["bbox_y1"] + out["bbox_y2"]) / 2.0) / H
    out["bbox_width"] = (out["bbox_x2"] - out["bbox_x1"]) / W
    out["bbox_height"] = (out["bbox_y2"] - out["bbox_y1"]) / H
    out["bbox_area"] = out["bbox_width"] * out["bbox_height"]

    # Pedestrian velocity and acceleration (frame-to-frame differences)
    group_keys = ["video_id", "pedestrian_id"]
    out["velocity_x"] = out.groupby(group_keys)["bbox_center_x"].diff().fillna(0.0)
    out["velocity_y"] = out.groupby(group_keys)["bbox_center_y"].diff().fillna(0.0)
    out["speed"] = np.sqrt(out["velocity_x"] ** 2 + out["velocity_y"] ** 2)
    out["acceleration_x"] = out.groupby(group_keys)["velocity_x"].diff().fillna(0.0)
    out["acceleration_y"] = out.groupby(group_keys)["velocity_y"].diff().fillna(0.0)

    return out


def read_ids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


# ----- DATA PREP -----
df = pd.DataFrame(features).copy()
df = df.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)
df = engineer_features(df)

# Frame-level binary target only
df = df[df["cross"].isin([0, 1])].copy()

# ----- DATA SPLIT -----
split_root = os.path.join("split_ids", "default")
train_videos = read_ids(os.path.join(split_root, "train.txt"))
val_videos = read_ids(os.path.join(split_root, "val.txt"))
test_videos = read_ids(os.path.join(split_root, "test.txt"))
assert set(train_videos).isdisjoint(set(val_videos)), "Train and validation video IDs overlap."
assert set(train_videos).isdisjoint(set(test_videos)), "Train and test video IDs overlap."
assert set(val_videos).isdisjoint(set(test_videos)), "Validation and test video IDs overlap."

train_df = df[df["video_id"].isin(train_videos)].copy()
val_df = df[df["video_id"].isin(val_videos)].copy()
test_df = df[df["video_id"].isin(test_videos)].copy()
assert len(set(train_df.index).intersection(val_df.index)) == 0, "Train and validation frame indices overlap."
assert len(set(train_df.index).intersection(test_df.index)) == 0, "Train and test frame indices overlap."
assert len(set(val_df.index).intersection(test_df.index)) == 0, "Validation and test frame indices overlap."

print("Frame counts by split:")
print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})

X_train = train_df[cat_cols + num_cols]
y_train = train_df["cross"]
X_val = val_df[cat_cols + num_cols]
y_val = val_df["cross"]
X_test = test_df[cat_cols + num_cols]
y_test = test_df["cross"]

# Handle imbalance using training split only
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
scale_pos_weight = float(neg) / float(pos) if pos > 0 else 1.0
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

# ----- MODEL TRAINING -----
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", StandardScaler(), num_cols)
])

clf = Pipeline([
    ("pre", pre),
    ("model", XGBClassifier(
        n_estimators=2500,
        max_depth=8,
        learning_rate=0.01,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
    ))
])

clf.fit(X_train, y_train)

# ----- EVALUATION -----
# Validation metrics
val_proba = clf.predict_proba(X_val)[:, 1]

val_pred = (val_proba >= DECISION_THRESHOLD).astype(int)
print(f"\nValidation threshold: {DECISION_THRESHOLD}")
print("Validation AUROC:", roc_auc_score(y_val, val_proba))


print(classification_report(y_val, val_pred))

# Test metrics
test_proba = clf.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= DECISION_THRESHOLD).astype(int)
print("Test AUROC:", roc_auc_score(y_test, test_proba))
print(classification_report(y_test, test_pred))

Frame counts by split:
{'train': 61805, 'val': 9583, 'test': 52966}
scale_pos_weight: 0.722

Validation threshold: 0.25
Validation AUROC: 0.9589167291460081
              precision    recall  f1-score   support

           0       0.95      0.81      0.88      4325
           1       0.86      0.96      0.91      5258

    accuracy                           0.90      9583
   macro avg       0.91      0.89      0.89      9583
weighted avg       0.90      0.90      0.90      9583

Test AUROC: 0.9064415235148972
              precision    recall  f1-score   support

           0       0.90      0.70      0.79     23243
           1       0.80      0.94      0.86     29723

    accuracy                           0.84     52966
   macro avg       0.85      0.82      0.83     52966
weighted avg       0.84      0.84      0.83     52966



In [ ]:
# Visualization of predictions on test video frames for XGBoost classifier
import pandas as pd
import cv2
import os

def predict_from_df(df, clf, cat_cols, num_cols, threshold, video_id=None, pedestrian_id=None):
    """Return per-frame crossing probabilities directly from the feature DataFrame."""
    data = df.copy()

    if video_id is not None:
        data = data[data["video_id"] == video_id].copy()
    if pedestrian_id is not None:
        data = data[data["pedestrian_id"] == pedestrian_id].copy()

    if data.empty:
        raise ValueError("No rows matched the requested video_id / pedestrian_id filter.")

    data = data.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)

    # Reuse the same feature logic used during model training
    if "bbox_center_x" not in data.columns:
        data = engineer_features(data)

    X = data.reindex(columns=cat_cols + num_cols, fill_value=0)
    scores = clf.predict_proba(X)[:, 1]
    result = data[[
        "video_id", "pedestrian_id", "frame_id",
        "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2",
        "cross"
    ]].copy()
    result["crossing_score"] = scores
    result["predicted_cross"] = (scores >= threshold).astype(int)
    return result


def write_prediction_video_from_df(df, obj, clf, cat_cols, num_cols, threshold, video_id, output_path, fps=30):
    """Write a video with bounding boxes, model prediction, and ground truth per frame."""
    predictions = predict_from_df(
        df, clf, cat_cols, num_cols, threshold=threshold, video_id=video_id
    )

    images_root = getattr(obj, "_images_path", os.path.join(".", "images"))
    video_path = os.path.join(images_root, video_id)
    if not os.path.isdir(video_path):
        raise FileNotFoundError(f"Video image folder not found: {video_path}")

    frame_files = sorted([
        f for f in os.listdir(video_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    if not frame_files:
        raise RuntimeError(f"No frames found in {video_path}. Expected .jpg or .png files.")

    first_frame = cv2.imread(os.path.join(video_path, frame_files[0]))
    if first_frame is None:
        raise RuntimeError(f"Could not read first frame: {frame_files[0]}")

    height, width = first_frame.shape[:2]
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    if not writer.isOpened():
        raise RuntimeError(f"Could not open video writer for: {output_path}")

    pred_by_frame = {}
    for _, row in predictions.iterrows():
        pred_by_frame.setdefault(int(row["frame_id"]), []).append(row)

    for frame_file in frame_files:
        frame_idx = int(os.path.splitext(frame_file)[0])
        img = cv2.imread(os.path.join(video_path, frame_file))
        if img is None:
            continue

        for row in pred_by_frame.get(frame_idx, []):
            bbox = [row["bbox_x1"], row["bbox_y1"], row["bbox_x2"], row["bbox_y2"]]
            score = float(row["crossing_score"])
            pred_label = int(row["predicted_cross"])
            gt_label = int(row["cross"])
            color = (0, int(255 * (1 - score)), int(255 * score))
            cv2.rectangle(img, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), color, 3)
            label = f"{row['pedestrian_id']} Pred:{pred_label} GT:{gt_label} Score:{score:.2f}"
            cv2.putText(
                img, label, (int(bbox[0]), max(20, int(bbox[1] - 10))),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2
            )

        writer.write(img)

    writer.release()
    return output_path


# Build videos for TEST split only using the same threshold as evaluation
test_ids_path = os.path.join("split_ids", "default", "test.txt")
test_video_ids = read_ids(test_ids_path)
output_dir = os.path.join(".", "predictions_test_videos")
os.makedirs(output_dir, exist_ok=True)

written = []
skipped = []
for video_id in test_video_ids:
    output_video_path = os.path.join(output_dir, f"{video_id}_predictions.mp4")
    try:
        write_prediction_video_from_df(
            df, jaad_api, clf, cat_cols, num_cols,
            threshold=DECISION_THRESHOLD,
            video_id=video_id,
            output_path=output_video_path
        )
        written.append(output_video_path)
        print(f"[OK] {video_id} -> {output_video_path}")
    except Exception as e:
        skipped.append((video_id, str(e)))
        print(f"[SKIP] {video_id}: {e}")

print(f"\nCreated {len(written)} test videos.")
print(f"Skipped {len(skipped)} test videos.")
if skipped:
    display(pd.DataFrame(skipped, columns=["video_id", "reason"]))

[OK] video_0005 -> .\predictions_test_videos\video_0005_predictions.mp4
[SKIP] video_0015: No rows matched the requested video_id / pedestrian_id filter.
[OK] video_0016 -> .\predictions_test_videos\video_0016_predictions.mp4
[OK] video_0017 -> .\predictions_test_videos\video_0017_predictions.mp4
[OK] video_0028 -> .\predictions_test_videos\video_0028_predictions.mp4
[SKIP] video_0036: No rows matched the requested video_id / pedestrian_id filter.
[OK] video_0042 -> .\predictions_test_videos\video_0042_predictions.mp4
[SKIP] video_0043: No rows matched the requested video_id / pedestrian_id filter.
[OK] video_0045 -> .\predictions_test_videos\video_0045_predictions.mp4
[OK] video_0046 -> .\predictions_test_videos\video_0046_predictions.mp4
[OK] video_0048 -> .\predictions_test_videos\video_0048_predictions.mp4
[OK] video_0053 -> .\predictions_test_videos\video_0053_predictions.mp4
[OK] video_0055 -> .\predictions_test_videos\video_0055_predictions.mp4
[SKIP] video_0058: No rows matched

,video_id,reason
0,video_0015,No rows matched the requested video_id / pedes...
1,video_0036,No rows matched the requested video_id / pedes...
2,video_0043,No rows matched the requested video_id / pedes...
3,video_0058,No rows matched the requested video_id / pedes...
4,video_0075,No rows matched the requested video_id / pedes...
5,video_0153,No rows matched the requested video_id / pedes...


In [ ]:
# Context extraction + cache (hybrid JAAD-first, model fallback)

import os
import time
import hashlib
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd


IMAGE_WIDTH = 1920.0
IMAGE_HEIGHT = 1080.0
IMAGE_DIAG = float(np.hypot(IMAGE_WIDTH, IMAGE_HEIGHT))

VEHICLE_LABELS = {
    "car", "truck", "bus", "motorcycle", "bicycle", "van", "tram", "train"
}
PEDESTRIAN_LABELS = {"person", "pedestrian"}


def _safe_to_int(x, default=0):
    try:
        return int(x)
    except Exception:
        return default


def _safe_bbox_xyxy(bbox):
    if bbox is None or len(bbox) != 4:
        return None
    x1, y1, x2, y2 = [float(v) for v in bbox]
    x1 = max(0.0, min(x1, IMAGE_WIDTH - 1))
    y1 = max(0.0, min(y1, IMAGE_HEIGHT - 1))
    x2 = max(0.0, min(x2, IMAGE_WIDTH - 1))
    y2 = max(0.0, min(y2, IMAGE_HEIGHT - 1))
    if x2 <= x1 or y2 <= y1:
        return None
    return [x1, y1, x2, y2]


def _label_from_raw(raw):
    if raw is None:
        return "unknown"
    if isinstance(raw, (int, float)):
        return str(int(raw))
    return str(raw).strip().lower().replace(" ", "_")


def _iter_annotation_items(maybe_dict):
    if isinstance(maybe_dict, dict):
        for k, v in maybe_dict.items():
            yield k, v
    elif isinstance(maybe_dict, list):
        for i, v in enumerate(maybe_dict):
            yield str(i), v


def _extract_rows_from_annotation_container(video_id, frame_id, container, source_name):
    rows = []
    if container is None:
        return rows

    for obj_id, obj in _iter_annotation_items(container):
        if not isinstance(obj, dict):
            continue

        cls = _label_from_raw(
            obj.get("class")
            or obj.get("label")
            or obj.get("type")
            or obj.get("category")
            or obj.get("object_type")
            or obj.get("obj_type")
        )

        frames = obj.get("frames")
        bboxes = obj.get("bbox") or obj.get("bboxes")

        if isinstance(frames, list) and isinstance(bboxes, list):
            frame_lookup = {int(f): b for f, b in zip(frames, bboxes)}
            bbox = frame_lookup.get(int(frame_id))
            bbox = _safe_bbox_xyxy(bbox) if bbox is not None else None
            if bbox is not None:
                rows.append({
                    "video_id": video_id,
                    "frame_id": int(frame_id),
                    "class": cls,
                    "x1": bbox[0],
                    "y1": bbox[1],
                    "x2": bbox[2],
                    "y2": bbox[3],
                    "score": 1.0,
                    "source": source_name,
                    "track_id": str(obj_id),
                })
            continue

        # Frame-wise dict format, e.g. {"123": [x1,y1,x2,y2]}
        bboxes_by_frame = obj.get("bboxes_by_frame") or obj.get("bbox_by_frame")
        if isinstance(bboxes_by_frame, dict):
            bbox = bboxes_by_frame.get(str(frame_id))
            if bbox is None:
                bbox = bboxes_by_frame.get(int(frame_id))
            bbox = _safe_bbox_xyxy(bbox) if bbox is not None else None
            if bbox is not None:
                rows.append({
                    "video_id": video_id,
                    "frame_id": int(frame_id),
                    "class": cls,
                    "x1": bbox[0],
                    "y1": bbox[1],
                    "x2": bbox[2],
                    "y2": bbox[3],
                    "score": 1.0,
                    "source": source_name,
                    "track_id": str(obj_id),
                })
            continue

        # Single-box format
        bbox = obj.get("bbox") or obj.get("box") or obj.get("xyxy")
        bbox = _safe_bbox_xyxy(bbox) if bbox is not None else None
        if bbox is not None:
            at_frame = _safe_to_int(obj.get("frame") if "frame" in obj else frame_id, frame_id)
            if at_frame == int(frame_id):
                rows.append({
                    "video_id": video_id,
                    "frame_id": int(frame_id),
                    "class": cls,
                    "x1": bbox[0],
                    "y1": bbox[1],
                    "x2": bbox[2],
                    "y2": bbox[3],
                    "score": float(obj.get("score", 1.0)),
                    "source": source_name,
                    "track_id": str(obj_id),
                })

    return rows


def _extract_jaad_context_objects(video_id, frame_id, video_obj):
    """Best-effort parser for JAAD-style object/traffic annotations."""
    rows = []

    candidate_keys = [
        "vehicle_annotations", "traffic_annotations", "traffic_objects",
        "objects", "obj_annotations", "annots", "annotations"
    ]
    for key in candidate_keys:
        container = video_obj.get(key)
        rows.extend(_extract_rows_from_annotation_container(video_id, frame_id, container, source_name=f"jaad:{key}"))

    return rows


def _extract_jaad_ped_rows(video_id, frame_id, video_obj):
    rows = []
    ped_annotations = video_obj.get("ped_annotations", {})
    if not isinstance(ped_annotations, dict):
        return rows

    for pid, ped in ped_annotations.items():
        if not isinstance(ped, dict):
            continue
        frames = ped.get("frames", [])
        bboxes = ped.get("bbox", [])
        if not frames or not bboxes:
            continue
        for fr, bb in zip(frames, bboxes):
            if int(fr) != int(frame_id):
                continue
            bb = _safe_bbox_xyxy(bb)
            if bb is None:
                continue
            rows.append({
                "video_id": video_id,
                "frame_id": int(frame_id),
                "class": "person",
                "x1": bb[0],
                "y1": bb[1],
                "x2": bb[2],
                "y2": bb[3],
                "score": 1.0,
                "source": "jaad:ped_annotations",
                "track_id": str(pid),
            })
    return rows


def _detect_crosswalk_heuristic(frame_bgr, road_mask=None):
    """Simple crosswalk heuristic used only when JAAD crosswalk geometry is unavailable."""
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)

    # Highlight bright stripe-like patterns
    _, bright = cv2.threshold(blur, 190, 255, cv2.THRESH_BINARY)
    edges = cv2.Canny(blur, 60, 150)
    cand = cv2.bitwise_and(bright, edges)

    if road_mask is not None:
        road_u8 = (road_mask.astype(np.uint8) * 255)
        cand = cv2.bitwise_and(cand, road_u8)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 3))
    cand = cv2.morphologyEx(cand, cv2.MORPH_CLOSE, kernel, iterations=2)
    cand = cv2.dilate(cand, kernel, iterations=1)

    crosswalk_mask = (cand > 0).astype(np.uint8)
    return crosswalk_mask


def _load_yolo_detector(model_name="yolov8m.pt"):
    from ultralytics import YOLO
    return YOLO(model_name)


def _load_segformer_cityscapes(model_id="nvidia/segformer-b5-finetuned-cityscapes-1024-1024"):
    from transformers import AutoImageProcessor, SegformerForSemanticSegmentation
    import torch

    processor = AutoImageProcessor.from_pretrained(model_id)
    model = SegformerForSemanticSegmentation.from_pretrained(model_id)
    model.eval()
    return processor, model, torch


def _infer_object_fallback_rows(video_id, frame_id, frame_bgr, yolo_model, conf=0.25, device=0):
    rows = []
    result = yolo_model.predict(frame_bgr, conf=conf, device=device, verbose=False)
    if not result:
        return rows

    r = result[0]
    if r.boxes is None or r.boxes.xyxy is None:
        return rows

    names = getattr(r, "names", {})
    boxes = r.boxes.xyxy.cpu().numpy()
    cls_ids = r.boxes.cls.cpu().numpy().astype(int)
    scores = r.boxes.conf.cpu().numpy()

    for bb, cid, score in zip(boxes, cls_ids, scores):
        label = _label_from_raw(names.get(int(cid), str(cid)))
        if label not in VEHICLE_LABELS and label not in PEDESTRIAN_LABELS:
            continue
        bb = _safe_bbox_xyxy(bb.tolist())
        if bb is None:
            continue
        rows.append({
            "video_id": video_id,
            "frame_id": int(frame_id),
            "class": label,
            "x1": bb[0],
            "y1": bb[1],
            "x2": bb[2],
            "y2": bb[3],
            "score": float(score),
            "source": "yolo_fallback",
            "track_id": "",
        })
    return rows


def _semantic_masks_from_segformer(frame_bgr, processor, seg_model, torch_mod, device="cpu"):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    inputs = processor(images=rgb, return_tensors="pt")
    if device != "cpu":
        inputs = {k: v.to(device) for k, v in inputs.items()}
        seg_model = seg_model.to(device)

    with torch_mod.no_grad():
        out = seg_model(**inputs)
        logits = out.logits
        pred = torch_mod.nn.functional.interpolate(
            logits,
            size=rgb.shape[:2],
            mode="bilinear",
            align_corners=False,
        ).argmax(dim=1)[0].cpu().numpy().astype(np.int32)

    id2label = {int(k): v.lower() for k, v in seg_model.config.id2label.items()}
    road_ids = [k for k, v in id2label.items() if "road" in v]
    sidewalk_ids = [k for k, v in id2label.items() if ("sidewalk" in v or "pavement" in v)]

    road_mask = np.isin(pred, road_ids).astype(np.uint8)
    sidewalk_mask = np.isin(pred, sidewalk_ids).astype(np.uint8)
    return road_mask, sidewalk_mask


def _extract_jaad_crosswalk_mask(video_obj, frame_id, shape_hw):
    """Best-effort polygon-to-mask parser for JAAD crosswalk annotations, if present."""
    h, w = shape_hw
    mask = np.zeros((h, w), dtype=np.uint8)

    candidate_keys = [
        "crosswalk_annotations", "crosswalks", "scene_annotations", "road_annotations"
    ]

    polygons = []
    for key in candidate_keys:
        ann = video_obj.get(key)
        if ann is None:
            continue

        if isinstance(ann, dict):
            per_frame = ann.get("polygons_by_frame") or ann.get("frame_polygons")
            if isinstance(per_frame, dict):
                polys = per_frame.get(str(frame_id)) or per_frame.get(int(frame_id)) or []
                polygons.extend(polys if isinstance(polys, list) else [])
            else:
                polys = ann.get("polygons") or ann.get("points")
                if isinstance(polys, list):
                    polygons.extend(polys)

        if isinstance(ann, list):
            polygons.extend(ann)

    valid = False
    for poly in polygons:
        if not isinstance(poly, (list, tuple)) or len(poly) < 3:
            continue
        pts = np.array(poly, dtype=np.int32)
        if pts.ndim != 2 or pts.shape[1] != 2:
            continue
        pts[:, 0] = np.clip(pts[:, 0], 0, w - 1)
        pts[:, 1] = np.clip(pts[:, 1], 0, h - 1)
        cv2.fillPoly(mask, [pts], color=1)
        valid = True

    return mask, valid



def build_frame_context_cache(video_ids, db, jaad_api, cache_dir="./cache", device=0, semantic_stride=1, conf=0.25, overwrite=False):
    """
    Build hybrid context cache:
      1) context objects table -> cache/context_objects.parquet
      2) semantic masks per video -> cache/context_semantics/{video_id}.npz

    Notes:
      - If overwrite=False, existing object rows and semantic files are reused.
      - If overwrite=True, targeted video_ids are regenerated.
    """
    cache_dir = Path(cache_dir)
    sem_dir = cache_dir / "context_semantics"
    sem_dir.mkdir(parents=True, exist_ok=True)

    objects_path = cache_dir / "context_objects.parquet"
    object_cols = ["video_id", "frame_id", "class", "x1", "y1", "x2", "y2", "score", "source", "track_id"]

    if objects_path.exists():
        existing_objects = pd.read_parquet(objects_path)
        for c in object_cols:
            if c not in existing_objects.columns:
                existing_objects[c] = np.nan
        existing_objects = existing_objects[object_cols].copy()
    else:
        existing_objects = pd.DataFrame(columns=object_cols)

    video_ids = [str(v) for v in video_ids]
    if overwrite and not existing_objects.empty:
        existing_objects = existing_objects[~existing_objects["video_id"].astype(str).isin(video_ids)].copy()

    existing_obj_videos = set(existing_objects["video_id"].astype(str).unique()) if not existing_objects.empty else set()

    # Lazy models, loaded only if needed.
    yolo_model = None
    segformer_bundle = None
    seg_device = "cuda" if (str(device) != "cpu") else "cpu"

    new_rows = []

    for i, vid in enumerate(video_ids, start=1):
        if vid not in db:
            print(f"[{i}/{len(video_ids)}] [SKIP] {vid}: missing in db")
            continue

        sem_path = sem_dir / f"{vid}.npz"
        reuse_semantics = sem_path.exists() and not overwrite
        reuse_objects = (vid in existing_obj_videos) and not overwrite

        if reuse_semantics and reuse_objects:
            print(f"[{i}/{len(video_ids)}] {vid}: reusing existing object + semantic cache")
            continue

        clip_path = os.path.join(jaad_api._clips_path, f"{vid}.mp4")
        cap = cv2.VideoCapture(clip_path)
        if not cap.isOpened():
            print(f"[{i}/{len(video_ids)}] [SKIP] {vid}: cannot open clip")
            continue

        video_obj = db[vid]

        # Only process frames where at least one pedestrian is annotated.
        target_frames = set()
        for _, ped in video_obj.get("ped_annotations", {}).items():
            for fr in ped.get("frames", []):
                target_frames.add(int(fr))

        if not target_frames:
            cap.release()
            print(f"[{i}/{len(video_ids)}] [SKIP] {vid}: no target frames")
            continue

        frame_idx = 0
        max_target = max(target_frames)
        semantic_records = []
        per_video_rows = []

        t0 = time.time()
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_idx > max_target:
                break
            if frame_idx not in target_frames:
                frame_idx += 1
                continue

            # Object rows: JAAD context + JAAD ped rows + detector fallback for missing context classes.
            if not reuse_objects:
                rows = []
                rows.extend(_extract_jaad_context_objects(vid, frame_idx, video_obj))
                rows.extend(_extract_jaad_ped_rows(vid, frame_idx, video_obj))

                existing_labels = {r["class"] for r in rows}
                needs_vehicle_fallback = not any(lbl in VEHICLE_LABELS for lbl in existing_labels)
                if needs_vehicle_fallback:
                    if yolo_model is None:
                        yolo_model = _load_yolo_detector("yolov8m.pt")
                    rows.extend(_infer_object_fallback_rows(vid, frame_idx, frame, yolo_model=yolo_model, conf=conf, device=device))

                per_video_rows.extend(rows)

            # Semantic masks (strictly current frame; no future smoothing).
            if (not reuse_semantics) and (frame_idx % int(max(1, semantic_stride)) == 0):
                jaad_crosswalk_mask, has_jaad_crosswalk = _extract_jaad_crosswalk_mask(video_obj, frame_idx, frame.shape[:2])

                road_mask = None
                sidewalk_mask = None
                semantic_source = "missing"

                try:
                    if segformer_bundle is None:
                        segformer_bundle = _load_segformer_cityscapes()
                    processor, seg_model, torch_mod = segformer_bundle
                    road_mask, sidewalk_mask = _semantic_masks_from_segformer(
                        frame, processor, seg_model, torch_mod, device=seg_device
                    )
                    semantic_source = "segformer_cityscapes"
                except Exception:
                    # Heuristic fallback: lower half as road, side margins as sidewalk-like areas.
                    h, w = frame.shape[:2]
                    road_mask = np.zeros((h, w), dtype=np.uint8)
                    sidewalk_mask = np.zeros((h, w), dtype=np.uint8)
                    road_mask[int(0.45 * h):, :] = 1
                    side_w = max(1, int(0.12 * w))
                    sidewalk_mask[int(0.35 * h):, :side_w] = 1
                    sidewalk_mask[int(0.35 * h):, w - side_w:] = 1
                    semantic_source = "heuristic"

                if has_jaad_crosswalk:
                    crosswalk_mask = jaad_crosswalk_mask.astype(np.uint8)
                    crosswalk_source = "jaad"
                    crosswalk_available = 1
                else:
                    crosswalk_mask = _detect_crosswalk_heuristic(frame, road_mask=road_mask)
                    crosswalk_source = "heuristic"
                    crosswalk_available = int(crosswalk_mask.sum() > 0)

                semantic_records.append({
                    "frame_id": int(frame_idx),
                    "road_mask": road_mask.astype(np.uint8),
                    "sidewalk_mask": sidewalk_mask.astype(np.uint8),
                    "crosswalk_mask": crosswalk_mask.astype(np.uint8),
                    "semantic_source": semantic_source,
                    "crosswalk_source": crosswalk_source,
                    "crosswalk_available": int(crosswalk_available),
                })

            frame_idx += 1

        cap.release()

        if not reuse_semantics:
            # Save per-video semantics in compressed form (downsampled).
            scale = 4
            if semantic_records:
                frame_ids = np.array([r["frame_id"] for r in semantic_records], dtype=np.int32)
                road_stack = np.stack([
                    cv2.resize(
                        r["road_mask"],
                        (max(1, r["road_mask"].shape[1] // scale), max(1, r["road_mask"].shape[0] // scale)),
                        interpolation=cv2.INTER_NEAREST,
                    )
                    for r in semantic_records
                ]).astype(np.uint8)
                sidewalk_stack = np.stack([
                    cv2.resize(
                        r["sidewalk_mask"],
                        (max(1, r["sidewalk_mask"].shape[1] // scale), max(1, r["sidewalk_mask"].shape[0] // scale)),
                        interpolation=cv2.INTER_NEAREST,
                    )
                    for r in semantic_records
                ]).astype(np.uint8)
                crosswalk_stack = np.stack([
                    cv2.resize(
                        r["crosswalk_mask"],
                        (max(1, r["crosswalk_mask"].shape[1] // scale), max(1, r["crosswalk_mask"].shape[0] // scale)),
                        interpolation=cv2.INTER_NEAREST,
                    )
                    for r in semantic_records
                ]).astype(np.uint8)

                semantic_source = np.array([r["semantic_source"] for r in semantic_records], dtype=object)
                crosswalk_source = np.array([r["crosswalk_source"] for r in semantic_records], dtype=object)
                crosswalk_available = np.array([r["crosswalk_available"] for r in semantic_records], dtype=np.uint8)

                np.savez_compressed(
                    sem_path,
                    frame_ids=frame_ids,
                    road_masks=road_stack,
                    sidewalk_masks=sidewalk_stack,
                    crosswalk_masks=crosswalk_stack,
                    semantic_source=semantic_source,
                    crosswalk_source=crosswalk_source,
                    crosswalk_available=crosswalk_available,
                    downsample_scale=np.array([scale], dtype=np.int32),
                )

        if per_video_rows:
            new_rows.extend(per_video_rows)

        fps = len(target_frames) / max(1e-6, (time.time() - t0))
        print(
            f"[{i}/{len(video_ids)}] {vid}: new_objects={len(per_video_rows)}, "
            f"semantic_frames={len(semantic_records) if not reuse_semantics else 'reused'}, "
            f"throughput={fps:.2f} target-frames/s"
        )

    new_df = pd.DataFrame(new_rows, columns=object_cols) if new_rows else pd.DataFrame(columns=object_cols)
    merged = pd.concat([existing_objects, new_df], ignore_index=True)
    if not merged.empty:
        merged = merged.drop_duplicates(
            subset=["video_id", "frame_id", "class", "x1", "y1", "x2", "y2", "source", "track_id"],
            keep="last",
        )

    merged.to_parquet(objects_path, index=False)
    print(f"[OK] wrote {len(merged)} object rows -> {objects_path}")



def _load_semantic_file(npz_path):
    x = np.load(npz_path, allow_pickle=True)
    return {
        "frame_ids": x["frame_ids"].astype(np.int32),
        "road_masks": x["road_masks"].astype(np.uint8),
        "sidewalk_masks": x["sidewalk_masks"].astype(np.uint8),
        "crosswalk_masks": x["crosswalk_masks"].astype(np.uint8),
        "semantic_source": x["semantic_source"],
        "crosswalk_source": x["crosswalk_source"],
        "crosswalk_available": x["crosswalk_available"].astype(np.uint8),
        "downsample_scale": int(x["downsample_scale"][0]),
    }


def load_frame_context(cache_dir="./cache"):
    """
    Returns:
        objects_df: columns [video_id, frame_id, class, x1, y1, x2, y2, score, source, track_id]
        semantics_by_video: dict[video_id] -> semantic arrays and metadata
    """
    cache_dir = Path(cache_dir)
    objects_path = cache_dir / "context_objects.parquet"
    sem_dir = cache_dir / "context_semantics"

    if not objects_path.exists():
        raise FileNotFoundError(f"Missing object cache: {objects_path}")

    objects_df = pd.read_parquet(objects_path)
    if "frame_id" in objects_df.columns:
        objects_df["frame_id"] = objects_df["frame_id"].astype(int)

    semantics_by_video = {}
    if sem_dir.exists():
        for npz_path in sorted(sem_dir.glob("*.npz")):
            vid = npz_path.stem
            semantics_by_video[vid] = _load_semantic_file(npz_path)

    return objects_df, semantics_by_video




In [ ]:
# Context feature engineering (row-level, strictly causal)

from sklearn.metrics import precision_recall_curve


def _frame_object_index(objects_df):
    idx = {}
    for (vid, fr), grp in objects_df.groupby(["video_id", "frame_id"]):
        idx[(vid, int(fr))] = grp
    return idx


def _distance_counts_to_objects(foot_xy, obj_df, radii_norm=(0.05, 0.10, 0.20)):
    if obj_df is None or len(obj_df) == 0:
        return {
            "nearest_vehicle_dist_norm": np.nan,
            "vehicle_count_r05": 0,
            "vehicle_count_r10": 0,
            "vehicle_count_r20": 0,
            "ctx_vehicle_missing": 1,
        }

    vmask = obj_df["class"].astype(str).str.lower().isin(VEHICLE_LABELS)
    veh = obj_df[vmask]
    if veh.empty:
        return {
            "nearest_vehicle_dist_norm": np.nan,
            "vehicle_count_r05": 0,
            "vehicle_count_r10": 0,
            "vehicle_count_r20": 0,
            "ctx_vehicle_missing": 1,
        }

    cx = ((veh["x1"].to_numpy() + veh["x2"].to_numpy()) / 2.0)
    cy = ((veh["y1"].to_numpy() + veh["y2"].to_numpy()) / 2.0)
    dx = cx - foot_xy[0]
    dy = cy - foot_xy[1]
    d = np.sqrt(dx * dx + dy * dy)
    d_norm = d / IMAGE_DIAG

    out = {
        "nearest_vehicle_dist_norm": float(np.min(d_norm)) if len(d_norm) else np.nan,
        "ctx_vehicle_missing": 0,
    }
    for r in radii_norm:
        out[f"vehicle_count_r{int(r * 100):02d}"] = int((d_norm <= r).sum())

    return out


def _compute_neighbor_features(df):
    """Compute nearby pedestrian features from same-frame JAAD pedestrian rows."""
    out_rows = []

    for (vid, fr), grp in df.groupby(["video_id", "frame_id"], sort=False):
        g = grp.copy()
        fx = g["feet_x"].to_numpy(dtype=float)
        fy = g["feet_y"].to_numpy(dtype=float)
        ids = g["pedestrian_id"].astype(str).to_numpy()

        if len(g) == 1:
            out_rows.append(pd.DataFrame({
                "_row_id": g["_row_id"].values,
                "neighbor_ped_count_r05": [0],
                "neighbor_ped_count_r10": [0],
                "nearest_ped_dist_norm": [1.5],
                "mean3_ped_dist_norm": [1.5],
            }))
            continue

        coords = np.stack([fx, fy], axis=1)
        dmat = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)

        # Exclude self via diagonal masking.
        np.fill_diagonal(dmat, np.inf)
        dnorm = dmat / IMAGE_DIAG

        c05 = (dnorm <= 0.05).sum(axis=1).astype(int)
        c10 = (dnorm <= 0.10).sum(axis=1).astype(int)
        nearest = np.min(dnorm, axis=1)

        sorted_d = np.sort(dnorm, axis=1)
        top3 = sorted_d[:, :3]
        # If fewer than 3 neighbors, pad with max norm cap.
        top3 = np.where(np.isfinite(top3), top3, 1.5)
        mean3 = top3.mean(axis=1)

        out_rows.append(pd.DataFrame({
            "_row_id": g["_row_id"].values,
            "neighbor_ped_count_r05": c05,
            "neighbor_ped_count_r10": c10,
            "nearest_ped_dist_norm": np.clip(nearest, 0.0, 1.5),
            "mean3_ped_dist_norm": np.clip(mean3, 0.0, 1.5),
        }))

    return pd.concat(out_rows, ignore_index=True) if out_rows else pd.DataFrame(columns=[
        "_row_id", "neighbor_ped_count_r05", "neighbor_ped_count_r10", "nearest_ped_dist_norm", "mean3_ped_dist_norm"
    ])


def _semantic_index_for_frame(frame_ids, frame_id):
    # strict-causal selection: latest available index <= frame_id
    pos = np.searchsorted(frame_ids, int(frame_id), side="right") - 1
    return int(pos) if pos >= 0 else -1


def _curb_boundary_mask(road_mask, sidewalk_mask):
    road = (road_mask > 0).astype(np.uint8)
    side = (sidewalk_mask > 0).astype(np.uint8)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    touch = cv2.bitwise_and(cv2.dilate(road, k, iterations=1), cv2.dilate(side, k, iterations=1))
    return (touch > 0).astype(np.uint8)


def _dist_transform_from_binary(target_mask):
    # Distance to nearest target pixel. If no target pixel exists, return None.
    if target_mask.sum() == 0:
        return None
    inv = (1 - (target_mask > 0).astype(np.uint8))
    return cv2.distanceTransform(inv, cv2.DIST_L2, 3)


def _semantic_features_for_point(video_id, frame_id, foot_x, foot_y, semantics_by_video, patch_radius=2):
    if video_id not in semantics_by_video:
        return {
            "feet_on_road": 0,
            "feet_on_sidewalk": 0,
            "feet_on_crosswalk": 0,
            "feet_semantic_conf": 0.0,
            "dist_to_curb_norm": 1.5,
            "dist_to_crosswalk_norm": 1.5,
            "in_crosswalk": 0,
            "ctx_semantic_missing": 1,
            "ctx_crosswalk_missing": 1,
        }

    s = semantics_by_video[video_id]
    idx = _semantic_index_for_frame(s["frame_ids"], frame_id)
    if idx < 0:
        return {
            "feet_on_road": 0,
            "feet_on_sidewalk": 0,
            "feet_on_crosswalk": 0,
            "feet_semantic_conf": 0.0,
            "dist_to_curb_norm": 1.5,
            "dist_to_crosswalk_norm": 1.5,
            "in_crosswalk": 0,
            "ctx_semantic_missing": 1,
            "ctx_crosswalk_missing": 1,
        }

    scale = int(s["downsample_scale"])
    road = s["road_masks"][idx].astype(np.uint8)
    sidewalk = s["sidewalk_masks"][idx].astype(np.uint8)
    crosswalk = s["crosswalk_masks"][idx].astype(np.uint8)

    mh, mw = road.shape[:2]
    x = int(np.clip(foot_x / scale, 0, mw - 1))
    y = int(np.clip(foot_y / scale, 0, mh - 1))

    y1 = max(0, y - patch_radius)
    y2 = min(mh, y + patch_radius + 1)
    x1 = max(0, x - patch_radius)
    x2 = min(mw, x + patch_radius + 1)

    patch_road = road[y1:y2, x1:x2]
    patch_side = sidewalk[y1:y2, x1:x2]
    patch_cross = crosswalk[y1:y2, x1:x2]

    total = max(1, patch_road.size)
    road_ratio = float((patch_road > 0).sum()) / total
    side_ratio = float((patch_side > 0).sum()) / total
    cross_ratio = float((patch_cross > 0).sum()) / total

    class_scores = {
        "road": road_ratio,
        "sidewalk": side_ratio,
        "crosswalk": cross_ratio,
        "other": max(0.0, 1.0 - max(road_ratio, side_ratio, cross_ratio)),
    }
    best_label = max(class_scores, key=class_scores.get)
    conf = float(class_scores[best_label])

    curb = _curb_boundary_mask(road, sidewalk)
    curb_dt = _dist_transform_from_binary(curb)
    cross_dt = _dist_transform_from_binary(crosswalk)

    mask_diag = float(np.hypot(mw, mh))
    dist_curb = float(curb_dt[y, x] / mask_diag) if curb_dt is not None else 1.5
    dist_cross = float(cross_dt[y, x] / mask_diag) if cross_dt is not None else 1.5

    in_cross = int(crosswalk[y, x] > 0)

    return {
        "feet_on_road": int(best_label == "road"),
        "feet_on_sidewalk": int(best_label == "sidewalk"),
        "feet_on_crosswalk": int(best_label == "crosswalk"),
        "feet_semantic_conf": conf,
        "dist_to_curb_norm": float(np.clip(dist_curb, 0.0, 1.5)),
        "dist_to_crosswalk_norm": float(np.clip(dist_cross, 0.0, 1.5)),
        "in_crosswalk": in_cross,
        "ctx_semantic_missing": 0,
        "ctx_crosswalk_missing": int(s["crosswalk_available"][idx] == 0),
    }


def engineer_context_features(df, db, context_cache_dir="./cache"):
    """
    Adds context features to per-frame pedestrian DataFrame.
    Strictly causal: uses only current frame context (or latest <= t for sparse semantic cache).
    """
    out = df.copy()
    out = out.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)
    out["_row_id"] = np.arange(len(out))

    out["feet_x"] = (out["bbox_x1"] + out["bbox_x2"]) / 2.0
    out["feet_y"] = out["bbox_y2"]

    objects_df, semantics_by_video = load_frame_context(context_cache_dir)
    obj_index = _frame_object_index(objects_df)

    # Vehicle proximity features (frame context objects)
    veh_rows = []
    for row in out[["video_id", "frame_id", "feet_x", "feet_y"]].itertuples(index=False):
        key = (row.video_id, int(row.frame_id))
        obj = obj_index.get(key)
        veh_rows.append(_distance_counts_to_objects((row.feet_x, row.feet_y), obj))

    veh_df = pd.DataFrame(veh_rows)
    out = pd.concat([out.reset_index(drop=True), veh_df.reset_index(drop=True)], axis=1)

    # Nearby pedestrian features from annotation table itself
    nb_df = _compute_neighbor_features(out[["_row_id", "video_id", "frame_id", "pedestrian_id", "feet_x", "feet_y"]])
    out = out.merge(nb_df, on="_row_id", how="left")

    # Semantic + curb/crosswalk features
    sem_rows = []
    for row in out[["video_id", "frame_id", "feet_x", "feet_y"]].itertuples(index=False):
        sem_rows.append(_semantic_features_for_point(
            video_id=row.video_id,
            frame_id=int(row.frame_id),
            foot_x=float(row.feet_x),
            foot_y=float(row.feet_y),
            semantics_by_video=semantics_by_video,
        ))

    sem_df = pd.DataFrame(sem_rows)
    out = pd.concat([out.reset_index(drop=True), sem_df.reset_index(drop=True)], axis=1)

    # Defensive fill for numeric stability.
    numeric_fill = {
        "nearest_vehicle_dist_norm": 1.5,
        "vehicle_count_r05": 0,
        "vehicle_count_r10": 0,
        "vehicle_count_r20": 0,
        "neighbor_ped_count_r05": 0,
        "neighbor_ped_count_r10": 0,
        "nearest_ped_dist_norm": 1.5,
        "mean3_ped_dist_norm": 1.5,
        "feet_semantic_conf": 0.0,
        "dist_to_curb_norm": 1.5,
        "dist_to_crosswalk_norm": 1.5,
        "in_crosswalk": 0,
        "ctx_vehicle_missing": 1,
        "ctx_semantic_missing": 1,
        "ctx_crosswalk_missing": 1,
    }
    for k, v in numeric_fill.items():
        if k in out.columns:
            out[k] = out[k].fillna(v)

    # Keep only needed helper columns for diagnostics.
    return out



In [ ]:
# Model integration + ablations

import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve
from xgboost import XGBClassifier


BASE_CAT_COLS = [
    "occlusion", "reaction", "hand_gesture", "look", "action", "nod",
    "age", "designated", "gender", "intersection",
    "motion_direction", "signalized", "traffic_direction"
]
BASE_NUM_COLS = [
    "bbox_center_x", "bbox_center_y", "bbox_width", "bbox_height", "bbox_area",
    "velocity_x", "velocity_y", "speed", "acceleration_x", "acceleration_y",
    "group_size", "num_lanes"
]

CONTEXT_GEOM_COLS = [
    "nearest_vehicle_dist_norm",
    "vehicle_count_r05", "vehicle_count_r10", "vehicle_count_r20",
    "neighbor_ped_count_r05", "neighbor_ped_count_r10",
    "nearest_ped_dist_norm", "mean3_ped_dist_norm",
]
CONTEXT_SEMANTIC_CAT_COLS = [
    "feet_on_road", "feet_on_sidewalk", "feet_on_crosswalk",
    "ctx_vehicle_missing", "ctx_semantic_missing", "ctx_crosswalk_missing",
]
CONTEXT_SEMANTIC_NUM_COLS = [
    "feet_semantic_conf", "dist_to_curb_norm", "dist_to_crosswalk_norm", "in_crosswalk"
]


def _read_ids(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]


def _split_df_by_video(df, split_root="split_ids/default"):
    train_videos = _read_ids(os.path.join(split_root, "train.txt"))
    val_videos = _read_ids(os.path.join(split_root, "val.txt"))
    test_videos = _read_ids(os.path.join(split_root, "test.txt"))

    assert set(train_videos).isdisjoint(set(val_videos))
    assert set(train_videos).isdisjoint(set(test_videos))
    assert set(val_videos).isdisjoint(set(test_videos))

    train_df = df[df["video_id"].isin(train_videos)].copy()
    val_df = df[df["video_id"].isin(val_videos)].copy()
    test_df = df[df["video_id"].isin(test_videos)].copy()

    return {
        "train": train_df,
        "val": val_df,
        "test": test_df,
        "train_videos": train_videos,
        "val_videos": val_videos,
        "test_videos": test_videos,
    }


def _build_pipeline(cat_cols, num_cols, scale_pos_weight=1.0):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", StandardScaler(), num_cols),
    ])

    clf = Pipeline([
        ("pre", pre),
        ("model", XGBClassifier(
            n_estimators=2500,
            max_depth=8,
            learning_rate=0.01,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
        )),
    ])
    return clf


def _tune_threshold_by_precision(y_true, proba, precision_target=0.85):
    p, r, t = precision_recall_curve(y_true, proba)
    # t has len n-1; align by dropping last precision/recall element.
    p2, r2 = p[:-1], r[:-1]
    valid = np.where(p2 >= precision_target)[0]
    if len(valid) == 0:
        # fallback: maximize F1 on validation
        f1 = (2 * p2 * r2) / np.maximum(1e-9, p2 + r2)
        best = int(np.argmax(f1))
        return float(t[best])
    # among valid thresholds, pick max recall
    best = valid[np.argmax(r2[valid])]
    return float(t[best])


def _eval_at_threshold(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    rep = classification_report(y_true, pred, output_dict=True, zero_division=0)
    return {
        "precision_0": rep["0"]["precision"],
        "recall_0": rep["0"]["recall"],
        "f1_0": rep["0"]["f1-score"],
        "precision_1": rep["1"]["precision"],
        "recall_1": rep["1"]["recall"],
        "f1_1": rep["1"]["f1-score"],
        "accuracy": rep["accuracy"],
    }


def run_ablation_experiments(df_with_context, splits, cat_cols, num_cols, threshold=0.25):
    """
    Runs three experiments:
      1) baseline
      2) baseline + geometry context
      3) full context

    Args:
      df_with_context: dataframe with base + context features
      splits: output dict from _split_df_by_video
      cat_cols, num_cols: full-context columns to use in experiment 3
      threshold: fallback threshold when validation tuning is disabled
    """
    train_df = splits["train"].copy()
    val_df = splits["val"].copy()
    test_df = splits["test"].copy()

    configs = {
        "baseline": {
            "cat": BASE_CAT_COLS,
            "num": BASE_NUM_COLS,
        },
        "baseline_plus_geometry_context": {
            "cat": BASE_CAT_COLS,
            "num": BASE_NUM_COLS + CONTEXT_GEOM_COLS,
        },
        "full_context": {
            "cat": cat_cols,
            "num": num_cols,
        },
    }

    all_rows = []

    for name, cfg in configs.items():
        use_cat = cfg["cat"]
        use_num = cfg["num"]

        X_train = train_df[use_cat + use_num]
        y_train = train_df["cross"].astype(int)
        X_val = val_df[use_cat + use_num]
        y_val = val_df["cross"].astype(int)
        X_test = test_df[use_cat + use_num]
        y_test = test_df["cross"].astype(int)

        neg = int((y_train == 0).sum())
        pos = int((y_train == 1).sum())
        spw = float(neg) / float(pos) if pos > 0 else 1.0

        clf = _build_pipeline(use_cat, use_num, scale_pos_weight=spw)
        clf.fit(X_train, y_train)

        val_proba = clf.predict_proba(X_val)[:, 1]
        tuned_thr = _tune_threshold_by_precision(y_val.to_numpy(), val_proba, precision_target=0.85)
        if not np.isfinite(tuned_thr):
            tuned_thr = float(threshold)

        test_proba = clf.predict_proba(X_test)[:, 1]

        val_metrics = _eval_at_threshold(y_val.to_numpy(), val_proba, tuned_thr)
        test_metrics = _eval_at_threshold(y_test.to_numpy(), test_proba, tuned_thr)

        all_rows.append({
            "experiment": name,
            "split": "val",
            "threshold": tuned_thr,
            "auroc": roc_auc_score(y_val, val_proba),
            **val_metrics,
        })
        all_rows.append({
            "experiment": name,
            "split": "test",
            "threshold": tuned_thr,
            "auroc": roc_auc_score(y_test, test_proba),
            **test_metrics,
        })

        print(f"\n=== {name} ===")
        print(f"threshold={tuned_thr:.4f}")
        print(f"val AUROC={all_rows[-2]['auroc']:.4f} | test AUROC={all_rows[-1]['auroc']:.4f}")

    return pd.DataFrame(all_rows)


# Full-context feature lists to use downstream.
FULL_CONTEXT_CAT_COLS = BASE_CAT_COLS + CONTEXT_SEMANTIC_CAT_COLS
FULL_CONTEXT_NUM_COLS = BASE_NUM_COLS + CONTEXT_GEOM_COLS + CONTEXT_SEMANTIC_NUM_COLS



In [ ]:
# Diagnostics and QA utilities

import os
import tempfile
import matplotlib.pyplot as plt


def context_coverage_report(df_with_context):
    report = {
        "rows": len(df_with_context),
        "vehicle_context_available_pct": float((1 - df_with_context["ctx_vehicle_missing"].mean()) * 100.0),
        "semantic_context_available_pct": float((1 - df_with_context["ctx_semantic_missing"].mean()) * 100.0),
        "crosswalk_context_available_pct": float((1 - df_with_context["ctx_crosswalk_missing"].mean()) * 100.0),
        "feet_on_road_pct": float(df_with_context["feet_on_road"].mean() * 100.0),
        "feet_on_sidewalk_pct": float(df_with_context["feet_on_sidewalk"].mean() * 100.0),
        "feet_on_crosswalk_pct": float(df_with_context["feet_on_crosswalk"].mean() * 100.0),
    }
    return pd.DataFrame([report])


def feature_distribution_by_split_and_class(df_with_context, split_root="split_ids/default"):
    splits = _split_df_by_video(df_with_context, split_root=split_root)

    def _tag(df_part, split_name):
        x = df_part.copy()
        x["split"] = split_name
        return x

    tagged = pd.concat([
        _tag(splits["train"], "train"),
        _tag(splits["val"], "val"),
        _tag(splits["test"], "test"),
    ], ignore_index=True)

    cols = [
        "nearest_vehicle_dist_norm", "vehicle_count_r10",
        "neighbor_ped_count_r10", "nearest_ped_dist_norm",
        "dist_to_curb_norm", "dist_to_crosswalk_norm", "feet_semantic_conf"
    ]

    stats = tagged.groupby(["split", "cross"])[cols].agg(["mean", "std", "median"]).reset_index()
    return stats


def visualize_context_overlay(video_id, frame_id, df_with_context, jaad_api, cache_dir="./cache", figsize=(12, 7)):
    objects_df, semantics_by_video = load_frame_context(cache_dir)

    img_path = os.path.join(getattr(jaad_api, "_images_path", "./images"), video_id, f"{int(frame_id):05d}.jpg")
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Could not read frame image: {img_path}")

    # Draw target pedestrians from df rows
    rows = df_with_context[(df_with_context["video_id"] == video_id) & (df_with_context["frame_id"].astype(int) == int(frame_id))]
    for _, r in rows.iterrows():
        x1, y1, x2, y2 = map(int, [r["bbox_x1"], r["bbox_y1"], r["bbox_x2"], r["bbox_y2"]])
        fx, fy = int(r["feet_x"]), int(r["feet_y"])
        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.circle(img, (fx, fy), 4, (255, 255, 0), -1)

    # Draw nearest vehicle line for first row if present
    if not rows.empty:
        first = rows.iloc[0]
        key = (video_id, int(frame_id))
        grp = objects_df[(objects_df["video_id"] == video_id) & (objects_df["frame_id"].astype(int) == int(frame_id))]
        veh = grp[grp["class"].astype(str).str.lower().isin(VEHICLE_LABELS)]
        if not veh.empty:
            fx, fy = float(first["feet_x"]), float(first["feet_y"])
            centers = np.column_stack([
                (veh["x1"].to_numpy() + veh["x2"].to_numpy()) / 2.0,
                (veh["y1"].to_numpy() + veh["y2"].to_numpy()) / 2.0,
            ])
            d = np.linalg.norm(centers - np.array([[fx, fy]]), axis=1)
            j = int(np.argmin(d))
            vx, vy = centers[j]
            cv2.line(img, (int(fx), int(fy)), (int(vx), int(vy)), (0, 0, 255), 2)

    # Blend semantic masks
    if video_id in semantics_by_video:
        s = semantics_by_video[video_id]
        idx = _semantic_index_for_frame(s["frame_ids"], int(frame_id))
        if idx >= 0:
            scale = int(s["downsample_scale"])
            road = s["road_masks"][idx]
            side = s["sidewalk_masks"][idx]
            cross = s["crosswalk_masks"][idx]

            h, w = img.shape[:2]
            road_up = cv2.resize(road, (w, h), interpolation=cv2.INTER_NEAREST)
            side_up = cv2.resize(side, (w, h), interpolation=cv2.INTER_NEAREST)
            cross_up = cv2.resize(cross, (w, h), interpolation=cv2.INTER_NEAREST)

            overlay = np.zeros_like(img)
            overlay[:, :, 1] = (road_up > 0).astype(np.uint8) * 120      # green road
            overlay[:, :, 0] = (side_up > 0).astype(np.uint8) * 120      # blue sidewalk
            overlay[:, :, 2] = (cross_up > 0).astype(np.uint8) * 180     # red crosswalk
            img = cv2.addWeighted(img, 1.0, overlay, 0.35, 0)

    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Context overlay | video={video_id}, frame={frame_id}")
    plt.axis("off")
    plt.show()


def _hash_df(df):
    payload = pd.util.hash_pandas_object(df.sort_values(df.columns.tolist()).reset_index(drop=True), index=True).values.tobytes()
    return hashlib.sha256(payload).hexdigest()


def _hash_npz_semantics(path):
    x = np.load(path, allow_pickle=True)
    h = hashlib.sha256()
    for k in sorted(x.files):
        h.update(k.encode("utf-8"))
        h.update(np.ascontiguousarray(x[k]).tobytes())
    return h.hexdigest()


def cache_consistency_check(sample_video_id, db, jaad_api, cache_dir="./cache", device=0):
    """
    Re-extracts one video into a temp cache and compares object/semantic hashes.
    """
    cache_dir = Path(cache_dir)
    obj_path = cache_dir / "context_objects.parquet"
    sem_path = cache_dir / "context_semantics" / f"{sample_video_id}.npz"

    if not obj_path.exists() or not sem_path.exists():
        raise FileNotFoundError("Run build_frame_context_cache first so baseline cache exists.")

    base_obj = pd.read_parquet(obj_path)
    base_obj = base_obj[base_obj["video_id"] == sample_video_id].copy()
    base_obj_hash = _hash_df(base_obj) if not base_obj.empty else "empty"
    base_sem_hash = _hash_npz_semantics(sem_path)

    with tempfile.TemporaryDirectory(prefix="ctx_consistency_") as td:
        build_frame_context_cache(
            [sample_video_id],
            db=db,
            jaad_api=jaad_api,
            cache_dir=td,
            device=device,
            semantic_stride=1,
            overwrite=True,
        )
        new_obj_path = Path(td) / "context_objects.parquet"
        new_sem_path = Path(td) / "context_semantics" / f"{sample_video_id}.npz"

        new_obj = pd.read_parquet(new_obj_path)
        new_obj_hash = _hash_df(new_obj) if not new_obj.empty else "empty"
        new_sem_hash = _hash_npz_semantics(new_sem_path)

    result = {
        "video_id": sample_video_id,
        "objects_hash_match": int(base_obj_hash == new_obj_hash),
        "semantics_hash_match": int(base_sem_hash == new_sem_hash),
        "base_object_hash": base_obj_hash,
        "new_object_hash": new_obj_hash,
        "base_semantic_hash": base_sem_hash,
        "new_semantic_hash": new_sem_hash,
    }
    return pd.DataFrame([result])



In [ ]:
# Recommended execution flow for the new context pipeline

# 1) Build baseline dataframe and engineered motion features if not already present.
# df = pd.DataFrame(features).copy()
# df = df.sort_values(["video_id", "pedestrian_id", "frame_id"]).reset_index(drop=True)
# df = engineer_features(df)
# df = df[df["cross"].isin([0, 1])].copy()

# 2) Build context cache for all split videos (run once; cached on disk).
# split_root = os.path.join("split_ids", "default")
# all_video_ids = _read_ids(os.path.join(split_root, "train.txt")) + _read_ids(os.path.join(split_root, "val.txt")) + _read_ids(os.path.join(split_root, "test.txt"))
# all_video_ids = sorted(set(all_video_ids))
# build_frame_context_cache(
#     video_ids=all_video_ids,
#     db=db,
#     jaad_api=jaad_api,
#     cache_dir="./cache",
#     device=0,
#     semantic_stride=1,
#     overwrite=False,
# )

# 3) Add context features to row-level frame dataframe.
# df_ctx = engineer_context_features(df, db=db, context_cache_dir="./cache")

# 4) Diagnostics and QA checks.
# display(context_coverage_report(df_ctx))
# display(feature_distribution_by_split_and_class(df_ctx, split_root="split_ids/default"))
# visualize_context_overlay(video_id="video_0223", frame_id=100, df_with_context=df_ctx, jaad_api=jaad_api, cache_dir="./cache")
# display(cache_consistency_check(sample_video_id="video_0223", db=db, jaad_api=jaad_api, cache_dir="./cache", device=0))

# 5) Run ablations and compare baseline vs context variants.
# splits = _split_df_by_video(df_ctx, split_root="split_ids/default")
# ablation_results = run_ablation_experiments(
#     df_with_context=df_ctx,
#     splits=splits,
#     cat_cols=FULL_CONTEXT_CAT_COLS,
#     num_cols=FULL_CONTEXT_NUM_COLS,
#     threshold=0.25,
# )
# display(ablation_results)

